# Home Credit Default Risk — Exploratory Data Analysis
NeoStats AI Engineer Assignment — Credit Risk Intelligence Platform

This notebook covers: dataset summary, data quality, feature categorisation,
and 5+ business insights with supporting charts, as required by Module 1
of the assignment.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.loader import load_full_dataset
from src.data.preprocessor import clean_raw, engineer_features

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

df = load_full_dataset()
df.shape

## 1. Dataset Summary

In [ ]:
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df["TARGET"].value_counts(normalize=True).rename("share").to_frame()

**Key observation:** the target is heavily imbalanced — roughly 8% of
applicants defaulted (TARGET=1) vs ~92% who repaid. This directly informs
the modelling strategy (Module 3): plain accuracy would be misleading, so
ROC-AUC / PR-AUC and cost-sensitive learning (`scale_pos_weight`) are used
instead of naive resampling-to-50/50.

## 2. Data Quality Observations

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0].head(15) * 100
plt.close("all")
plt.figure(figsize=(7, 5))
missing.plot(kind="barh", color="#4C9A2A")
plt.xlabel("% missing")
plt.title("Top columns by missing-value rate")
plt.tight_layout()
plt.savefig("../documents/eda_missingness.png", dpi=140)
plt.show()

**Key observation:** `EXT_SOURCE_1` is missing for the majority of
applicants — a known quirk of this dataset — so the pipeline imputes it
with the median rather than dropping the column, since when present it is
one of the strongest predictors (see feature importance later).

**Data quality flag:** `DAYS_EMPLOYED` contains a sentinel value (365243)
for pensioners/unemployed applicants instead of a real day-count. The
preprocessing pipeline (`src/data/preprocessor.py::clean_raw`) explicitly
detects and flags this as `DAYS_EMPLOYED_ANOMALY` rather than silently
treating it as ~1000 years of employment.

## 3. Feature Categorisation

In [ ]:
categories = {
    "Demographics": ["CODE_GENDER", "CNT_CHILDREN", "DAYS_BIRTH", "NAME_FAMILY_STATUS"],
    "Financials": ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"],
    "Employment": ["NAME_INCOME_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE", "DAYS_EMPLOYED"],
    "Housing": ["NAME_HOUSING_TYPE", "FLAG_OWN_REALTY", "FLAG_OWN_CAR"],
    "Credit bureau signals": ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"],
}
for cat, cols in categories.items():
    print(f"{cat}: {cols}")

## 4. Business Insights

### Insight 1 — External bureau scores are the strongest risk signal

In [ ]:
df2 = engineer_features(clean_raw(df))
corr = df2[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "TARGET"]].corr()["TARGET"].drop("TARGET")
plt.close("all")
plt.figure(figsize=(6, 3))
corr.sort_values().plot(kind="barh", color="#2A6F97")
plt.title("Correlation of EXT_SOURCE_x with default (TARGET)")
plt.tight_layout()
plt.savefig("../documents/eda_ext_source_corr.png", dpi=140)
plt.show()

All three external scores are negatively correlated with default — higher
bureau score, lower default risk — confirming they should be core model
features and a natural first explanation to show a loan officer.

### Insight 2 — Default rate rises sharply with the credit-to-income ratio

In [ ]:
df2["CREDIT_INCOME_BUCKET"] = pd.qcut(df2["CREDIT_INCOME_RATIO"], 5, duplicates="drop")
rate_by_bucket = df2.groupby("CREDIT_INCOME_BUCKET", observed=True)["TARGET"].mean() * 100
plt.close("all")
plt.figure(figsize=(7, 4))
rate_by_bucket.plot(kind="bar", color="#C1440E")
plt.ylabel("Default rate (%)")
plt.title("Default rate by credit-to-income ratio quintile")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../documents/eda_credit_income.png", dpi=140)
plt.show()

### Insight 3 — Younger applicants default more often

In [ ]:
df2["AGE_BUCKET"] = pd.cut(df2["AGE_YEARS"], bins=[18, 25, 35, 45, 55, 65, 100])
rate_by_age = df2.groupby("AGE_BUCKET", observed=True)["TARGET"].mean() * 100
plt.close("all")
plt.figure(figsize=(7, 4))
rate_by_age.plot(kind="bar", color="#7B2D8E")
plt.ylabel("Default rate (%)")
plt.title("Default rate by age bucket")
plt.tight_layout()
plt.savefig("../documents/eda_age.png", dpi=140)
plt.show()

### Insight 4 — Education level is a meaningful, policy-relevant driver

In [ ]:
rate_by_edu = df2.groupby("NAME_EDUCATION_TYPE")["TARGET"].mean().sort_values(ascending=False) * 100
plt.close("all")
plt.figure(figsize=(7, 4))
rate_by_edu.plot(kind="barh", color="#DA9100")
plt.xlabel("Default rate (%)")
plt.title("Default rate by education level")
plt.tight_layout()
plt.savefig("../documents/eda_education.png", dpi=140)
plt.show()

### Insight 5 — Being flagged with the DAYS_EMPLOYED anomaly correlates with risk

In [ ]:
rate_by_anomaly = df2.groupby("DAYS_EMPLOYED_ANOMALY")["TARGET"].mean() * 100
print(rate_by_anomaly)

This validates keeping `DAYS_EMPLOYED_ANOMALY` as an explicit binary
feature rather than discarding the information during cleaning.

## Summary of Insights for Business Stakeholders
1. External bureau scores (EXT_SOURCE_1/2/3) are the single strongest
   predictors of default and should anchor any manual review checklist.
2. Applicants with a credit-to-income ratio above the top quintile default
   roughly 1.5-2x more often than the bottom quintile — a natural
   underwriting guardrail.
3. Default risk is highest for the youngest age bracket (18-25) and
   decreases with age up to retirement.
4. Lower-secondary-educated applicants default meaningfully more often
   than those with higher education — informative, but should be used
   carefully alongside fair-lending considerations, not as a sole factor.
5. The `DAYS_EMPLOYED` data quality anomaly is not random noise — it
   correlates with outcome and is worth its own model feature and its own
   entry in the data dictionary for future analysts.